# Link movement bouts to neuronal activity

Reload the bout tables produced by `analyse_movement_bouts.ipynb` from
`data/derived/movement_bouts/`. Run its save cell before using this notebook.

Run this notebook with `movement_bouts/` as the kernel's working directory.
Both notebooks can import helper functions from the adjacent `utils.py`.

The saved tables retain each headstage entry. Use `idx` to look up its animal,
headstage, source session, and feature file in `session_info.loc[idx]`.
`parameters` records the extraction settings, export time, and Git state.


# 1. Load movement bouts data

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

# Run with movement_bouts/ as the working directory, alongside utils.py.
data_dir = Path("../../data").resolve()
results_dir = data_dir / "derived" / "movement_bouts"

movement_bouts = pd.read_pickle(results_dir / "movement_bouts.pkl")
immobility_bouts = pd.read_pickle(results_dir / "immobility_bouts.pkl")
session_info = pd.read_pickle(results_dir / "session_info.pkl")
parameters = json.loads((results_dir / "parameters.json").read_text(encoding="utf-8"))
DT = parameters["DT"]

print(f"Loaded {len(movement_bouts)} movement bouts across "
      f"{len(session_info)} session entries.")
display(parameters)
display(session_info.head())


# 2. Make PCA (first for one idx then we will build the helper function etc.)

In [ ]:
# Input parameters. Premotor durations is counted back from movement onset. i.e. from 0s to minus PREMOTOR_DURATION_S seconds.
DT = parameters["DT"]
PREMOTOR_DURATION_S = 0.5
N_PREMOTOR_BINS = round(PREMOTOR_DURATION_S / DT)

assert abs(N_PREMOTOR_BINS * DT - PREMOTOR_DURATION_S) < 1e-10


In [ ]:
import pickle

idx = 0
info = session_info.loc[idx]
session = int(info["session"])

# Select this session from its source collection.
neural_file = info["feature_file"].replace("_feature_ss", "_data_ss")
neuron_ids_file = info["feature_file"].replace("_feature_ss", "_neuron_idx_ss")

with (data_dir / neural_file).open("rb") as f:
    neural_data = pickle.load(f)[session]
with (data_dir / neuron_ids_file).open("rb") as f:
    neuron_ids = pickle.load(f)[session]

session_bouts = movement_bouts.loc[movement_bouts["idx"] == idx]

assert neural_data.ndim == 2
assert neural_data.shape[0] == len(neuron_ids)
assert neural_data.shape[1] == info["n_bins"]

print(f"idx={idx}: {info['animal']}, headstage {info['headstage']}, session {session}")
print(f"Neural data: {neural_data.shape} (neurons, time bins)")
print(f"Movement bouts: {len(session_bouts)}")


In [ ]:
import numpy as np

onset_bins = session_bouts["onset_bin"].to_numpy(dtype=int)
start_bins = onset_bins - N_PREMOTOR_BINS

# Skip bouts without a complete premotor window or a recorded onset.
assert N_PREMOTOR_BINS > 0
valid_window = (start_bins >= 0) & (onset_bins < neural_data.shape[1])

excluded_bouts = session_bouts.loc[~valid_window].copy()
excluded_bouts["exclusion_reason"] = np.where(
    start_bins[~valid_window] < 0,
    "Premotor window starts before recording",
    "Movement onset is outside recording",
)
valid_bouts = session_bouts.loc[valid_window]
start_bins = start_bins[valid_window]
onset_bins = onset_bins[valid_window]
bout_ids = valid_bouts.index.to_numpy(copy=True)

# Keep original 5 ms spike counts; the movement-onset bin is excluded.
if len(valid_bouts):
    premotor_spike_counts = np.stack([
        neural_data[:, start:onset]
        for start, onset in zip(start_bins, onset_bins)
    ])
else:
    premotor_spike_counts = np.empty(
        (0, len(neuron_ids), N_PREMOTOR_BINS), dtype=neural_data.dtype
    )
relative_bin_starts_s = np.arange(-N_PREMOTOR_BINS, 0) * DT

assert premotor_spike_counts.shape == (
    len(bout_ids), len(neuron_ids), N_PREMOTOR_BINS
)

print(f"Skipped {len(excluded_bouts)} of {len(session_bouts)} bouts.")
print(f"Premotor counts: {premotor_spike_counts.shape} (bouts, neurons, time bins)")
print(f"Window: [{relative_bin_starts_s[0]:g}, 0) s; bin width: {DT:g} s")
